# ActorAttack & X-Teaming — Design D Hidden State Extraction

Post-hoc Design D forward passes on saved ActorAttack and X-Teaming conversations.
Produces hidden states compatible with the Crescendo faithful representations for
cross-framework transfer analysis.

## Context reconstruction

Unlike Crescendo, neither ActorAttack nor X-Teaming uses rollback. Every turn
accumulates in the target's context.

```
Context for turn T = [all user/asst pairs from turns 1..T-1]
                   + [user message at turn T]
                   + add_generation_prompt=True
```

So `n_context_turns = turn_idx - 1` always.

## Outputs

```
data/representations/actorattack_harmful_faithful/
data/representations/actorattack_benign_faithful/
data/representations/xteaming_harmful_faithful/
data/representations/xteaming_benign_faithful/
    hidden_states.npy    ← float16, shape (N, N_LAYERS, 4096)
    metadata.parquet     ← one row per turn
```

**Metadata columns:** `pair_id, attempt, turn_idx, n_context_turns,
final_verdict, label, framework, split, fname`

In [ ]:
import os

GPU_IDS = [4, 5, 6, 7]
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in GPU_IDS)
LOGICAL_GPU_IDS = list(range(len(GPU_IDS)))

import json
import sys
import threading
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

MODEL_ID      = "meta-llama/Llama-3.1-8B-Instruct"
DTYPE         = torch.bfloat16
LAYER_INDICES = [1, 16, 32]   # first, middle, last — matches Crescendo faithful

CONV_ROOT = repo_root / "data" / "conversations"
REPR_ROOT = repo_root / "data" / "representations"

print(f"GPUs:   {GPU_IDS} → logical {LOGICAL_GPU_IDS}")
print(f"Layers: {LAYER_INDICES}")

## Helpers

In [ ]:
def build_contexts(turns: list[dict]) -> list[dict]:
    """
    Build Design D context for each turn.
    No rollback: all prior turns stay in context.

    Returns list of dicts:
        turn_idx        — 1-based
        messages        — chat messages for apply_chat_template
        n_context_turns — turn_idx - 1
    """
    by_idx = {}
    for t in turns:
        by_idx.setdefault(t["turn_idx"], {})[t["role"]] = t

    accumulated = []
    result = []

    for turn_idx in sorted(by_idx):
        pair   = by_idx[turn_idx]
        user_t = pair.get("user")
        asst_t = pair.get("assistant")
        if not user_t or not asst_t:
            continue

        messages = list(accumulated) + [{"role": "user", "content": user_t["content"]}]

        result.append({
            "turn_idx":        turn_idx,
            "messages":        messages,
            "n_context_turns": len(accumulated) // 2,
        })

        # All turns accumulate — no rollback
        accumulated.append({"role": "user",      "content": user_t["content"]})
        accumulated.append({"role": "assistant", "content": asst_t["content"]})

    return result


@torch.no_grad()
def forward_design_d(model, tokenizer, messages: list[dict]) -> np.ndarray:
    """Design D: hidden state at last position with add_generation_prompt=True."""
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)

    with torch.autocast(device_type="cuda", dtype=DTYPE):
        out = model(input_ids, output_hidden_states=True)

    layer_vecs = np.stack([
        out.hidden_states[l][0, -1, :].cpu().to(torch.float16).numpy()
        for l in LAYER_INDICES
    ])  # (N_LAYERS, 4096)

    del out, input_ids
    return layer_vecs


def extract_conversation(
    model, tokenizer, conv: dict, fpath: Path, framework: str, split: str
) -> tuple[np.ndarray | None, list[dict]]:
    turns = conv.get("turns", [])
    if not turns:
        return None, []

    contexts = build_contexts(turns)
    if not contexts:
        return None, []

    vecs = []
    rows = []
    for ctx in contexts:
        vec = forward_design_d(model, tokenizer, ctx["messages"])
        vecs.append(vec)
        rows.append(dict(
            pair_id         = conv["objective_pair_id"],
            attempt         = conv.get("attempt", 1),
            turn_idx        = ctx["turn_idx"],
            n_context_turns = ctx["n_context_turns"],
            final_verdict   = conv["verdict"],
            label           = 1 if split == "harmful" else 0,
            framework       = framework,
            split           = split,
            fname           = fpath.name,
        ))

    return np.stack(vecs).astype(np.float16), rows


print("Helpers defined.")

## Parallel extraction infrastructure

In [ ]:
def load_done(save_dir: Path):
    meta_path = save_dir / "metadata.parquet"
    if meta_path.exists():
        meta = pd.read_parquet(meta_path)
        done = set(zip(meta["pair_id"], meta["attempt"]))
        print(f"  Resuming: {len(done)} conversations already done")
        return done, [np.load(str(save_dir / "hidden_states.npy"))], [meta]
    return set(), [], []


def save_results(all_states, all_meta, save_dir: Path):
    save_dir.mkdir(parents=True, exist_ok=True)
    states = np.concatenate(all_states, axis=0).astype(np.float16)
    meta   = pd.concat(all_meta, ignore_index=True)
    np.save(str(save_dir / "hidden_states.npy"), states)
    meta.to_parquet(save_dir / "metadata.parquet", index=False)
    print(f"Saved {states.shape[0]} vectors → {save_dir.name}")
    print(f"  Shape: {states.shape}  ({states.nbytes/1e9:.2f} GB)")


def run_extraction(conv_dir: Path, save_dir: Path, framework: str, split: str):
    files = sorted(conv_dir.glob("*.json"))
    print(f"\n{'='*60}")
    print(f"{framework} / {split}: {len(files)} conversations")

    done_set, ex_states, ex_meta = load_done(save_dir)

    # Read pair_id/attempt without loading full JSON for resume check
    pending = []
    for f in files:
        conv = json.loads(f.read_text())
        key  = (conv["objective_pair_id"], conv.get("attempt", 1))
        if key not in done_set:
            pending.append(f)

    print(f"  Pending: {len(pending)} / {len(files)}")
    if not pending:
        print("  Nothing to do.")
        return

    n_gpus  = len(LOGICAL_GPU_IDS)
    chunks  = [pending[i::n_gpus] for i in range(n_gpus)]

    all_states = list(ex_states)
    all_meta   = list(ex_meta)
    lock = threading.Lock()
    pbar = tqdm(total=len(pending), desc=f"{framework}/{split}",
                file=sys.stdout, dynamic_ncols=True)

    def worker(gpu_id, chunk):
        torch.cuda.set_device(gpu_id)
        device = f"cuda:{gpu_id}"
        print(f"GPU {gpu_id}: loading model...", flush=True)
        tok = AutoTokenizer.from_pretrained(MODEL_ID)
        mdl = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, torch_dtype=DTYPE, device_map={"": device}
        )
        mdl.eval()
        print(f"GPU {gpu_id}: ready, {len(chunk)} conversations", flush=True)

        local_states, local_meta = [], []
        for fpath in chunk:
            conv = json.loads(fpath.read_text())
            try:
                states, rows = extract_conversation(mdl, tok, conv, fpath, framework, split)
                if states is not None:
                    local_states.append(states)
                    local_meta.append(pd.DataFrame(rows))
            except Exception as e:
                print(f"GPU {gpu_id} ERROR {fpath.name}: {e}", flush=True)
            with lock:
                pbar.update(1)

        with lock:
            all_states.extend(local_states)
            all_meta.extend(local_meta)

        del mdl
        torch.cuda.empty_cache()
        print(f"GPU {gpu_id}: done", flush=True)

    threads = [
        threading.Thread(target=worker, args=(gpu_id, chunks[gpu_id]))
        for gpu_id in LOGICAL_GPU_IDS
    ]
    for t in threads: t.start()
    for t in threads: t.join()
    pbar.close()

    if all_states:
        save_results(all_states, all_meta, save_dir)
    else:
        print("No states collected — check for errors above.")


print("Infrastructure defined.")

## Run all four datasets

Each cell is independent — run separately or all at once.
Resume-safe: already-extracted conversations are skipped.

In [ ]:
run_extraction(
    conv_dir  = CONV_ROOT / "actorattack_harmful",
    save_dir  = REPR_ROOT / "actorattack_harmful_faithful",
    framework = "actorattack",
    split     = "harmful",
)

In [ ]:
run_extraction(
    conv_dir  = CONV_ROOT / "actorattack_benign",
    save_dir  = REPR_ROOT / "actorattack_benign_faithful",
    framework = "actorattack",
    split     = "benign",
)

In [ ]:
run_extraction(
    conv_dir  = CONV_ROOT / "xteaming_harmful_v2",
    save_dir  = REPR_ROOT / "xteaming_harmful_faithful",
    framework = "xteaming",
    split     = "harmful",
)

In [ ]:
run_extraction(
    conv_dir  = CONV_ROOT / "xteaming_benign_v3",
    save_dir  = REPR_ROOT / "xteaming_benign_faithful",
    framework = "xteaming",
    split     = "benign",
)

## Verification

In [ ]:
datasets = [
    ("actorattack", "harmful", "actorattack_harmful_faithful"),
    ("actorattack", "benign",  "actorattack_benign_faithful"),
    ("xteaming",    "harmful", "xteaming_harmful_faithful"),
    ("xteaming",    "benign",  "xteaming_benign_faithful"),
]

for fw, split, folder in datasets:
    d = REPR_ROOT / folder
    if not (d / "hidden_states.npy").exists():
        print(f"{folder}: NOT FOUND")
        continue

    states = np.load(str(d / "hidden_states.npy"), mmap_mode="r")
    meta   = pd.read_parquet(d / "metadata.parquet")

    print(f"=== {fw} / {split} ===")
    print(f"  Shape:    {states.shape}  ({states.nbytes/1e9:.2f} GB)")
    print(f"  Turns:    {len(meta)}")
    print(f"  Verdicts: {meta['final_verdict'].value_counts().to_dict()}")
    print(f"  Context depth range: {meta['n_context_turns'].min()}–{meta['n_context_turns'].max()}")

    sample = states[:min(200, len(states)), -1, :].astype(np.float32)
    print(f"  Layer 32 stats: mean={sample.mean():.4f}  std={sample.std():.4f}  "
          f"NaNs={np.isnan(sample).sum()}  Infs={np.isinf(sample).sum()}")
    print()